### Build Constructors Dimension (`dim_constructors`)

This notebook builds the **Constructors Dimension Table** in the gold layer of our Formula 1 data lakehouse. It is part of the **medallion architecture** pipeline (bronze -> silver -> gold) and sits in the `04-gold` folder.

##### What this notebook does:
1. **Reads** the `constructors` table from the silver layer and the `ref_nationalaty_regions` reference table from the gold layer
2. **Joins** them on `nationality` to enrich each constructor with its geographic region
3. **Selects** only the columns needed for the dimension: constructor_id, constructor_name, nationality, and region
4. **Writes** the result as a Delta table to `formula1.gold.dim_constructors`

##### Purpose:
The `dim_constructors` table serves as a denormalized dimension table that allows analysts and dashboards to query constructor (team) information enriched with region data - without needing to perform joins at query time.

##### Dependencies:
- **Upstream config:** **`00-common/01.environment-config`** (provides `catalog_name`, `silver_schema`, `gold_schema` variables)
- **Source tables:** `formula1.silver.constructors`, `formula1.gold.ref_nationalaty_regions`
- **Target table:** `formula1.gold.dim_constructors`

In [0]:
%run ../00-common/01.environment-config 

##### Step 0: Import Required Functions
Here we import all the built-in functions from `pyspark.sql.functions` (such as `col`, `lit`, `when`, `concat`, etc.). This gives us access to all the transformation and aggregation functions we might need when working with our DataFrames later in the notebook.

In [0]:
from pyspark.sql.functions import *

##### Define the Gold Target Table
We define the fully qualified target table name where our final dimension table will be written. The variable `target_table` is set to `formula1.gold.dim_constructors` by combining:
- `catalog_name` -> the Unity Catalog name (`formula1`)
- `gold_schema` -> the gold layer schema (`gold`)
- Table name -> `dim_constructors`

This follows the **medallion architecture** (bronze -> silver -> gold), where the gold layer holds clean, business-ready dimension and fact tables.

In [0]:

target_table = f'{catalog_name}.{gold_schema}.dim_constructors'

##### Step 1: Read the Source Tables
We read two tables into Spark DataFrames:
- `constructors_df` -> reads `formula1.silver.constructors` which contains constructor/team details (constructor_id, constructor_name, nationality)
- `ref_nationality_region_df` -> reads `formula1.gold.ref_nationalaty_regions` which is a reference table mapping each nationality to its geographic region (e.g., "British" -> "Europe", "Mexican" -> "North America")

Both DataFrames are loaded lazily (Spark won't actually read the data until an action like `display()` or `write` is triggered). The `nationality` column exists in both tables and will be used as the join key in the next step.

In [0]:
constructors_df = spark.table(f'{catalog_name}.{silver_schema}.constructors')
ref_nationality_region_df = spark.read.table(f'{catalog_name}.{gold_schema}.ref_nationalaty_regions')

##### Step 2: Join Constructors with Nationality-Region Mapping
We create a new DataFrame called `dim_constructors_df` by performing a **left outer join** between `constructors_df` and `ref_nationality_region_df` on the `nationality` column. This enriches each constructor with its geographic region.

**Join condition:** `constructors_df.nationality == ref_nationality_region_df.nationality`

**Join type:** Left Outer - ensures ALL constructors are kept even if no matching nationality is found in the reference table (region would be `null` in that case).

**Selected columns from the joined result:**
| Column | Source DataFrame | Description |
|--------|-----------------|-------------|
| `constructor_id` | constructors_df | Unique identifier for the constructor/team |
| `constructor_name` | constructors_df | Name of the team (e.g., "Red Bull", "Ferrari") |
| `nationality` | constructors_df | Nationality of the constructor (e.g., "British", "Italian") |
| `region` | ref_nationality_region_df | Geographic region (e.g., "Europe", "Asia", "North America") |

In [0]:
dim_constructors_df =(
    constructors_df
     .join(
        ref_nationality_region_df,
        constructors_df.nationality == ref_nationality_region_df.nationality,
        'left_outer')
    .select(
        constructors_df.constructor_id,
        constructors_df.constructor_name,
        constructors_df.nationality,
        ref_nationality_region_df.region.alias('nationality_region'))
)

##### Step 3: Write to Gold Layer
Finally, we write the `dim_constructors_df` DataFrame to the gold schema as a Delta table called `dim_constructors`. This table serves as a **dimension table** that combines constructor identity with geographic region into a single, denormalized table ready for analytics and reporting.

- **Format:** Delta (supports ACID transactions, time travel, and schema evolution)
- **Mode:** SaveAsTable (creates or replaces the table)
- **Target:** `formula1.gold.dim_constructors`

In [0]:
(
    dim_constructors_df
    .write
    .mode('overwrite')
    .option('overwriteSchema', True)
    .format('delta')
    .saveAsTable(target_table)
)

In [0]:
display(spark.table(target_table))

### Entity Relationship Diagram

The diagram below shows how the **source tables** relate to each other and how they combine into the **target dimension table** (gold layer).

```
┌─────────────────────────────────────┐         ┌─────────────────────────────────────────────┐
│       SILVER LAYER (Source)         │         │           GOLD LAYER (Target)               │
├─────────────────────────────────────┤         ├─────────────────────────────────────────────┤
│                                     │         │                                             │
│  ┌───────────────────────────┐      │         │  ┌───────────────────────────────────────┐  │
│  │  silver.constructors      │      │         │  │       gold.dim_constructors            │  │
│  ├───────────────────────────┤      │         │  ├───────────────────────────────────────┤  │
│  │ PK constructor_id         │      │         │  │  constructor_id   (from constructors)  │  │
│  │    constructor_name       │      │  JOIN   │  │  constructor_name (from constructors)  │  │
│  │    nationality ───────────│──┐   │ ──────► │  │  nationality      (from constructors)  │  │
│  │    ingestion_timestamp    │  │   │         │  │  region           (from ref table)     │  │
│  │    source_file            │  │   │         │  └───────────────────────────────────────┘  │
│  └───────────────────────────┘  │   │         │                                             │
│                                  │   │         │                                             │
└──────────────────────────────────┼───┘         └─────────────────────────────────────────────┘
                                   │
         ┌─────────────────────────┼─────┐
         │  GOLD LAYER (Reference) │     │
         ├─────────────────────────┼─────┤
         │                         │     │
         │  ┌──────────────────────┴──┐  │
         │  │ gold.ref_nationalaty_   │  │
         │  │ regions                 │  │
         │  ├─────────────────────────┤  │
         │  │    nationality ─────────│──┘
         │  │    region               │
         │  └─────────────────────────┘
         │                               │
         └───────────────────────────────┘
```

**Relationship:** `constructors.nationality` (FK) → `ref_nationalaty_regions.nationality` (PK) — **LEFT OUTER JOIN**

**Result:** One row per constructor, enriched with geographic region. Constructors without a matching nationality retain `null` for region.